# SE(3)-Invariant Graph Neural Network for Molecular Property Prediction

**Author:** Nandini  
**Stack:** PyTorch, NumPy, matplotlib  
**Topics:** E(n)-equivariant message passing (EGNN-style), distance-based invariant features, molecular graph construction, QM9-style property prediction from scratch

---

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
torch.manual_seed(42)
np.random.seed(42)
print(f'PyTorch {torch.__version__}')

PyTorch 2.11.0+cu130


## 1. Molecular Graph Construction
Convert SMILES to 3D molecular graphs with atom features and coordinates.

In [2]:
# One-hot encode atomic number
ATOM_LIST = [6, 7, 8, 9, 15, 16, 17, 35]  # C, N, O, F, P, S, Cl, Br

def atom_features(atom):
    z = atom.GetAtomicNum()
    one_hot = [1 if z == a else 0 for a in ATOM_LIST]
    one_hot.append(1 if z not in ATOM_LIST else 0)  # other
    return one_hot

def mol_to_graph(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    res = AllChem.EmbedMolecule(mol, params)
    if res != 0:
        return None
    AllChem.MMFFOptimizeMolecule(mol)
    mol = Chem.RemoveHs(mol)
    
    # Get 3D coordinates for heavy atoms only
    conf = mol.GetConformer()
    n_atoms = mol.GetNumAtoms()
    
    # Node features
    h = [atom_features(mol.GetAtomWithIdx(i)) for i in range(n_atoms)]
    h = torch.tensor(h, dtype=torch.float32)
    
    # Coordinates
    pos = torch.tensor([list(conf.GetAtomPosition(i)) for i in range(n_atoms)], dtype=torch.float32)
    
    # Fully connected edges (for small molecules)
    edge_index = []
    for i in range(n_atoms):
        for j in range(n_atoms):
            if i != j:
                edge_index.append([i, j])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t()
    
    return h, pos, edge_index, n_atoms

# Test
test_graph = mol_to_graph('c1ccccc1')  # benzene
print(f'Benzene: {test_graph[3]} atoms, node features {test_graph[0].shape}, edges {test_graph[2].shape}')

Benzene: 6 atoms, node features torch.Size([6, 9]), edges torch.Size([2, 30])


## 2. E(n)-Invariant Message Passing Layer

Following the EGNN paradigm (Satorras et al., 2021):
- Messages depend on **pairwise distances** $\|x_i - x_j\|^2$ (invariant under rotations/translations)
- Node features update via aggregated messages
- No coordinate update (invariant prediction, not equivariant dynamics)

In [3]:
class InvariantMPLayer(nn.Module):
    """E(n)-invariant message passing layer.
    Messages depend on pairwise distances and node features.
    """
    def __init__(self, node_dim, hidden_dim=64):
        super().__init__()
        # Message MLP: h_i || h_j || d_ij^2 -> message
        self.msg_mlp = nn.Sequential(
            nn.Linear(2 * node_dim + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )
        # Update MLP: h_i || aggregated_msg -> h_i'
        self.update_mlp = nn.Sequential(
            nn.Linear(node_dim + hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, node_dim),
        )
        self.node_dim = node_dim
    
    def forward(self, h, pos, edge_index):
        src, dst = edge_index[0], edge_index[1]
        
        # Pairwise squared distances (SE(3)-invariant)
        diff = pos[src] - pos[dst]
        dist_sq = (diff ** 2).sum(dim=-1, keepdim=True)  # [n_edges, 1]
        
        # Compute messages
        msg_input = torch.cat([h[src], h[dst], dist_sq], dim=-1)
        messages = self.msg_mlp(msg_input)  # [n_edges, hidden]
        
        # Aggregate (sum)
        n_nodes = h.shape[0]
        agg = torch.zeros(n_nodes, messages.shape[1], device=h.device)
        agg.index_add_(0, dst, messages)
        
        # Update
        h_new = self.update_mlp(torch.cat([h, agg], dim=-1))
        return h + h_new  # Residual connection


class InvariantGNN(nn.Module):
    """SE(3)-invariant GNN for scalar molecular property prediction."""
    def __init__(self, in_dim=9, hidden_dim=64, n_layers=3):
        super().__init__()
        self.embed = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([
            InvariantMPLayer(hidden_dim, hidden_dim) for _ in range(n_layers)
        ])
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, h, pos, edge_index, batch_idx=None):
        h = self.embed(h)
        for layer in self.layers:
            h = layer(h, pos, edge_index)
        
        # Global mean pooling
        if batch_idx is not None:
            n_graphs = batch_idx.max().item() + 1
            graph_emb = torch.zeros(n_graphs, h.shape[1], device=h.device)
            counts = torch.zeros(n_graphs, 1, device=h.device)
            graph_emb.index_add_(0, batch_idx, h)
            counts.index_add_(0, batch_idx, torch.ones(h.shape[0], 1, device=h.device))
            graph_emb = graph_emb / counts
        else:
            graph_emb = h.mean(dim=0, keepdim=True)
        
        return self.readout(graph_emb)

model = InvariantGNN()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params}')

Model parameters: 79553


## 3. Verify SE(3)-Invariance
Apply random rotation + translation, verify output is identical.

In [4]:
def random_rotation_matrix():
    """Random SO(3) rotation via QR decomposition."""
    A = torch.randn(3, 3)
    Q, R = torch.linalg.qr(A)
    Q = Q * torch.sign(torch.diag(R))  # Ensure det = +1
    if torch.det(Q) < 0:
        Q[:, 0] *= -1
    return Q

h, pos, edge_index, n = mol_to_graph('CC(=O)OC1=CC=CC=C1C(=O)O')  # Aspirin

model.eval()
with torch.no_grad():
    out_original = model(h, pos, edge_index)
    
    # Random rotation + translation
    R = random_rotation_matrix()
    t = torch.randn(1, 3) * 10  # large translation
    pos_transformed = (pos @ R.T) + t
    
    out_transformed = model(h, pos_transformed, edge_index)

diff = torch.abs(out_original - out_transformed).item()
print(f'Output (original coords):    {out_original.item():.8f}')
print(f'Output (rotated+translated): {out_transformed.item():.8f}')
print(f'Absolute difference:         {diff:.2e}')
print(f'\nSE(3)-invariance verified: {"YES" if diff < 1e-5 else "NO"} (diff = {diff:.2e})')

Output (original coords):    -0.10356027
Output (rotated+translated): -0.10356026
Absolute difference:         7.45e-09

SE(3)-invariance verified: YES (diff = 7.45e-09)


## 4. Train on LogP Prediction
Predicting computed LogP for a set of drug molecules.

In [5]:
smiles_dataset = [
    'CC(=O)OC1=CC=CC=C1C(=O)O', 'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O',
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C', 'CC(=O)NC1=CC=C(O)C=C1',
    'COC1=CC2=CC(=CC2=CC1)C(C)C(=O)O', 'OC(=O)CC1=CC=CC=C1NC1=C(Cl)C=CC=C1Cl',
    'CCCCCCCC', 'C1CCCCC1', 'C1=CC=CC=C1', 'CCO', 'CCCCCC', 'CC(C)O',
    'OC1=CC=CC=C1', 'CC(=O)O', 'CCOCC', 'CC(C)(C)O',
    'C1=CC=C(C=C1)O', 'CC=CC', 'CCCC(=O)O', 'C1CCOC1',
]

graphs = []
targets = []
for smi in smiles_dataset:
    g = mol_to_graph(smi)
    if g is not None:
        mol = Chem.MolFromSmiles(smi)
        logp = Descriptors.MolLogP(mol)
        graphs.append(g)
        targets.append(logp)

targets = torch.tensor(targets, dtype=torch.float32)
print(f'Built {len(graphs)} molecular graphs')
print(f'LogP range: [{targets.min():.2f}, {targets.max():.2f}]')

# Train
model = InvariantGNN()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
model.train()

losses = []
for epoch in range(300):
    total_loss = 0
    for i, (h, pos, edge_index, n_atoms) in enumerate(graphs):
        optimizer.zero_grad()
        pred = model(h, pos, edge_index)
        loss = F.mse_loss(pred.squeeze(), targets[i])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(graphs)
    losses.append(avg_loss)
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:4d} | MSE: {avg_loss:.4f}')

Built 20 molecular graphs
LogP range: [-1.03, 4.36]


Epoch   50 | MSE: 3.0339


Epoch  100 | MSE: 1.2164


Epoch  150 | MSE: 0.6391


Epoch  200 | MSE: 1.2741


Epoch  250 | MSE: 1.2114


Epoch  300 | MSE: 0.5523


In [6]:
# Predictions vs targets
model.eval()
preds = []
with torch.no_grad():
    for h, pos, edge_index, _ in graphs:
        p = model(h, pos, edge_index).item()
        preds.append(p)
preds = np.array(preds)
tgts = targets.numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].semilogy(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.2)

# Parity plot
axes[1].scatter(tgts, preds, c='#ef4444', s=40, alpha=0.8, edgecolors='white', linewidth=0.5)
lims = [min(tgts.min(), preds.min()) - 0.5, max(tgts.max(), preds.max()) + 0.5]
axes[1].plot(lims, lims, 'k--', alpha=0.4)
axes[1].set_xlabel('True LogP (RDKit)')
axes[1].set_ylabel('Predicted LogP (EGNN)')
axes[1].set_title(f'LogP Prediction (R² = {np.corrcoef(tgts, preds)[0,1]**2:.3f})')
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()